# 🎹 Piano MP3 → MIDI
**시작 전:** 런타임 → 런타임 유형 변경 → **T4 GPU**  ·  이후 `Ctrl+F9` 한 번으로 끝.

**1단계** — 설치 (약 30~40초)

In [ ]:
# ── 설치 (재시작 없음, 약 30~40초) ───────────────────────
import subprocess, os

has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
whl     = 'cu121' if has_gpu else 'cpu'

subprocess.run('pip install uv -q', shell=True, capture_output=True)

pkgs = [
    f'torch torchaudio --index-url https://download.pytorch.org/whl/{whl}',
    'librosa==0.9.2',
    'piano_transcription_inference',
    'pretty_midi',
    'noisereduce',
    'soundfile',
]
# 병렬 설치 (의존성 충돌 없는 것들 한꺼번에)
standalone = 'pretty_midi noisereduce soundfile librosa==0.9.2'
subprocess.run(f'uv pip install {standalone} -q', shell=True, capture_output=True)
subprocess.run(f'uv pip install torch torchaudio '
               f'--index-url https://download.pytorch.org/whl/{whl} -q',
               shell=True, capture_output=True)
subprocess.run('uv pip install piano_transcription_inference -q',
               shell=True, capture_output=True)

print('✅ 설치 완료  |  디바이스:', 'GPU ⚡' if has_gpu else 'CPU 🐢')

**2단계** — MP3 여러 개 업로드 → 순서대로 자동 변환 → 각각 다운로드

In [ ]:
import os, glob, subprocess
import numpy as np, librosa, soundfile as sf, noisereduce as nr
import pretty_midi, torch
from google.colab import files
from piano_transcription_inference import PianoTranscription, sample_rate, load_audio

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── 파일 전부 업로드 (Ctrl+클릭으로 여러 개 선택) ─────────
print('📂 변환할 MP3를 모두 선택하세요 (Ctrl+클릭으로 여러 개 동시 선택)...')
uploaded = files.upload()
queue    = list(uploaded.keys())
print(f'\n총 {len(queue)}개 접수:')
for i, name in enumerate(queue, 1):
    print(f'  {i}. {name}')

# ── 모델 1회만 로드 (이후 재사용) ────────────────────────
print('\n🤖 AI 모델 로드 중...')
transcriptor = PianoTranscription(device=DEVICE)
print(f'✅ 준비 완료 ({DEVICE.upper()}) — 변환 시작\n')
print('=' * 48)

# ── 변환 함수 ─────────────────────────────────────────────
def convert_one(src_name):
    src  = '/content/' + src_name
    safe = ''.join(c if c.isalnum() or c in ' _-' else '_'
                   for c in src_name.rsplit('.', 1)[0])[:50]
    wav  = '/content/audio.wav'
    proc = '/content/proc.wav'
    raw  = '/content/raw.mid'
    out  = f'/content/{safe}.mid'

    for f in [wav, proc, raw]:
        if os.path.exists(f): os.remove(f)

    subprocess.run(f'ffmpeg -y -i "{src}" "{wav}" -loglevel quiet', shell=True)

    y, sr = librosa.load(wav, sr=44100, mono=True)
    y, _  = librosa.effects.trim(y, top_db=35)
    y     = nr.reduce_noise(y=y, sr=sr, y_noise=y[:int(sr*0.3)],
                            stationary=False, prop_decrease=0.75)
    rms   = np.sqrt(np.mean(y**2))
    y     = np.clip(y * np.clip(10**(-18/20)/(rms+1e-9), 0.1, 10.0), -1.0, 1.0)
    sf.write(proc, y, 44100)

    audio, _ = load_audio(proc, sr=sample_rate, mono=True)
    transcriptor.transcribe(audio, raw)

    midi_raw   = pretty_midi.PrettyMIDI(raw)
    midi_clean = pretty_midi.PrettyMIDI(initial_tempo=midi_raw.estimate_tempo())
    kept = removed = 0
    for inst in midi_raw.instruments:
        ni    = pretty_midi.Instrument(program=inst.program, is_drum=inst.is_drum)
        notes = sorted(inst.notes, key=lambda n: (n.start, n.pitch))
        vt    = max(10, np.percentile([n.velocity for n in notes], 5)) if notes else 10
        seen  = {}
        for n in notes:
            if n.end-n.start < 0.04 or n.velocity < vt \
            or not (21 <= n.pitch <= 108) \
            or (n.pitch in seen and n.start-seen[n.pitch] < 0.05):
                removed += 1; continue
            ni.notes.append(pretty_midi.Note(
                velocity=int(np.clip(n.velocity,1,127)),
                pitch=n.pitch, start=n.start, end=min(n.end, n.start+8.0)))
            seen[n.pitch] = n.start; kept += 1
        ni.control_changes = inst.control_changes
        midi_clean.instruments.append(ni)

    midi_clean.write(out)
    return out, kept

# ── 순서대로 자동 처리 ────────────────────────────────────
results = []
for i, name in enumerate(queue, 1):
    print(f'[{i}/{len(queue)}] {name}')
    try:
        out, kept = convert_one(name)
        print(f'  ✅ {kept:,}개 음표 → 다운로드 중...')
        files.download(out)
        results.append((name, kept, '✅'))
    except Exception as e:
        print(f'  ❌ 오류: {e}')
        results.append((name, 0, '❌'))
    print()

# ── 결과 요약 ─────────────────────────────────────────────
print('=' * 48)
for name, kept, status in results:
    print(f'  {status}  {name[:38]}  ({kept:,}개)')
print('=' * 48)